<h1 style="text-align:center; color:#00F1FA;">Utility Chains Overview</h1>
<p style="text-align:center; color:#00F1FA;">Covers two practical utility chains in LangChain - load_summarize_chain for summarising documents and LLMRequestsChain for fetching and extracting information from the web.</p>

<h2 style="color:#FA00EE;">API Keys Setup</h2>
<p style="color:#FA00EE;">Loading API keys securely from the environment file. Never hardcode keys directly in the notebook.</p>

In [1]:
import os
from dotenv import load_dotenv

# Load keys from .env file
load_dotenv(dotenv_path="/Users/kpitsiakkos/Documents/Langchain-and-Langgraph-Specialization/Utility Chains/.env")

# Verify key loaded
print("OPENAI KEY FOUND:", bool(os.getenv("OPENAI_API_KEY")))

OPENAI KEY FOUND: True


<h2 style="color:#FA00EE;">Imports</h2>
<p style="color:#FA00EE;">Importing all required libraries for summarisation and HTTP request chains.</p>

In [2]:
# LLM and prompt
from langchain_openai import OpenAI
from langchain_core.prompts import PromptTemplate

# Summarisation chain
from langchain_classic.chains.summarize import load_summarize_chain

# Text splitting and document creation
from langchain_text_splitters import CharacterTextSplitter
from langchain_core.documents import Document

# HTTP requests chain
from langchain_classic.chains import LLMRequestsChain, LLMChain

/Users/kpitsiakkos/Documents/Langchain-and-Langgraph-Specialization/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm

A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.2.6 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/runpy.py", line 196, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "/Library/Frameworks/Python.framework/Versions/3

<hr>
<h2 style="color:#00F1FA;">Part 1 - Summarising Documents</h2>
<p style="color:#00F1FA;">load_summarize_chain reads a large document, splits it into chunks, summarises each chunk separately, and then combines all summaries into one final summary. This solves the token limit problem - instead of sending the whole document to the LLM at once, it processes it in pieces.</p>

In [3]:
# Initialize OpenAI LLM with higher temperature for more creative summaries
llm = OpenAI(
    temperature=0.9,
    openai_api_key=os.getenv("OPENAI_API_KEY")
)

In [4]:
# Read the document from disk
with open("sample.txt") as f:
    data = f.read()

print(f"Document loaded — {len(data)} characters")

Document loaded — 8901 characters


<h3 style="color:#00F1FA;">Split Text into Chunks</h3>
<p style="color:#00F1FA;">Large documents must be split into smaller chunks before summarisation. LLMs have a token limit so we cannot send the entire document at once. Each chunk is wrapped in a Document object which is the standard LangChain format for passing text to chains.</p>

In [5]:
# Split the raw text into manageable chunks
text_splitter = CharacterTextSplitter()
texts = text_splitter.split_text(data)

# Wrap each chunk in a Document object — required format for LangChain chains
docs = [Document(page_content=t) for t in texts]

print(f"Total chunks created: {len(docs)}")

Total chunks created: 3


In [6]:
# Preview the document chunks
docs

[Document(metadata={}, page_content='Technology is the application of scientific knowledge and practical tools to solve problems, improve processes, and enhance the quality of human life. From the invention of the wheel to the development of artificial intelligence, technology has been the driving force behind human progress and civilisation. It encompasses a vast range of fields including computing, communications, medicine, energy, transportation, and manufacturing.\n\nThe history of technology stretches back thousands of years. Early humans developed primitive tools from stone and bone, marking the beginning of the Stone Age. The invention of the printing press by Johannes Gutenberg in the 15th century revolutionised the spread of knowledge and literacy. The Industrial Revolution of the 18th and 19th centuries transformed societies from agrarian economies to industrialised nations, powered by steam engines, mechanised factories, and railways.\n\nThe 20th century witnessed an unprece

<h3 style="color:#00F1FA;">Run the Summarisation Chain</h3>
<p style="color:#00F1FA;">The map_reduce strategy works in two stages. First it sends each chunk to the LLM separately and gets a summary for each one. Then it combines all those summaries and sends them to the LLM one more time to produce a single final summary. This handles documents of any length.</p>

In [7]:
# Create the summarisation chain using map_reduce strategy
# map_reduce = summarise each chunk first, then combine all summaries into one
# verbose=True shows intermediate steps so you can see each chunk being summarised
chain = load_summarize_chain(llm, chain_type="map_reduce", verbose=True)

# Run the chain on all document chunks
chain.invoke(docs)



> Entering new MapReduceDocumentsChain chain...


> Entering new LLMChain chain...
Prompt after formatting:
Write a concise summary of the following:


"Technology is the application of scientific knowledge and practical tools to solve problems, improve processes, and enhance the quality of human life. From the invention of the wheel to the development of artificial intelligence, technology has been the driving force behind human progress and civilisation. It encompasses a vast range of fields including computing, communications, medicine, energy, transportation, and manufacturing.

The history of technology stretches back thousands of years. Early humans developed primitive tools from stone and bone, marking the beginning of the Stone Age. The invention of the printing press by Johannes Gutenberg in the 15th century revolutionised the spread of knowledge and literacy. The Industrial Revolution of the 18th and 19th centuries transformed societies from agrarian economies to industrial

{'input_documents': [Document(metadata={}, page_content='Technology is the application of scientific knowledge and practical tools to solve problems, improve processes, and enhance the quality of human life. From the invention of the wheel to the development of artificial intelligence, technology has been the driving force behind human progress and civilisation. It encompasses a vast range of fields including computing, communications, medicine, energy, transportation, and manufacturing.\n\nThe history of technology stretches back thousands of years. Early humans developed primitive tools from stone and bone, marking the beginning of the Stone Age. The invention of the printing press by Johannes Gutenberg in the 15th century revolutionised the spread of knowledge and literacy. The Industrial Revolution of the 18th and 19th centuries transformed societies from agrarian economies to industrialised nations, powered by steam engines, mechanised factories, and railways.\n\nThe 20th century 